In [ ]:
from IPython.display import HTML, display

html = r"""
<!DOCTYPE html>
<html>
<head>

<style>

#chessGame {
    font-family: Arial, sans-serif;
    text-align: center;
    margin-top: 20px;
}

#chessBoard {
    width: 480px;
    height: 480px;
    margin: 20px auto;
    display: grid;
    grid-template-columns: repeat(8, 1fr);
    border: 4px solid #222;
}

.square {
    width: 60px;
    height: 60px;
    display: flex;
    justify-content: center;
    align-items: center;
    font-size: 42px;
    cursor: pointer;
    user-select: none;
    box-sizing: border-box;
}

.light {
    background: #f0d9b5;
}

.dark {
    background: #b58863;
}

.selected {
    outline: 4px solid #00aaff;
    outline-offset: -4px;
}

.legal {
    box-shadow: inset 0 0 0 5px rgba(0, 180, 0, 0.35);
}

.capture {
    box-shadow: inset 0 0 0 5px rgba(220, 0, 0, 0.45);
}

#turnText {
    font-size: 22px;
    font-weight: bold;
    margin: 10px;
}

#statusText {
    font-size: 20px;
    font-weight: bold;
    margin: 10px;
}

#messageText {
    font-size: 18px;
    margin: 10px;
}

.clocks {
    display: flex;
    justify-content: center;
    gap: 30px;
    margin: 15px;
}

.clockBox {
    background: #222;
    color: white;
    padding: 10px 25px;
    border-radius: 10px;
    min-width: 120px;
}

.clockLabel {
    font-size: 16px;
}

.clockTime {
    font-size: 30px;
    font-weight: bold;
    margin-top: 5px;
}

#restartButton {
    padding: 10px 20px;
    font-size: 16px;
    border: none;
    border-radius: 8px;
    background: #333;
    color: white;
    cursor: pointer;
}

#restartButton:hover {
    background: #555;
}

#promotionMenu {
    display: none;
    position: fixed;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    background: white;
    border: 3px solid #222;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 5px 25px rgba(0,0,0,0.4);
    z-index: 1000;
}

#promotionMenu h3 {
    margin-top: 0;
}

.promotionButton {
    font-size: 40px;
    padding: 10px 15px;
    margin: 5px;
    cursor: pointer;
    border: 2px solid #aaa;
    border-radius: 8px;
    background: #f5f5f5;
}

.promotionButton:hover {
    background: #ddd;
}

</style>

</head>

<body>

<div id="chessGame">

    <h1>♟️ Chess Game ♟️</h1>

    <div class="clocks">

        <div class="clockBox">
            <div class="clockLabel">♔ White</div>
            <div id="whiteClock" class="clockTime">10:00</div>
        </div>

        <div class="clockBox">
            <div class="clockLabel">♚ Black</div>
            <div id="blackClock" class="clockTime">10:00</div>
        </div>

    </div>

    <div id="turnText">White's Turn</div>

    <div id="statusText"></div>

    <div id="messageText"></div>

    <div id="chessBoard"></div>

    <button id="restartButton" onclick="restartGame()">
        🔄 Restart Game
    </button>

</div>


<div id="promotionMenu">

    <h3 id="promotionPlayer">Choose Promotion</h3>

    <button class="promotionButton" onclick="finishPromotion('queen')">
        ♛
    </button>

    <button class="promotionButton" onclick="finishPromotion('rook')">
        ♜
    </button>

    <button class="promotionButton" onclick="finishPromotion('bishop')">
        ♝
    </button>

    <button class="promotionButton" onclick="finishPromotion('knight')">
        ♞
    </button>

</div>


<script>

/* =========================================================
   INITIAL BOARD
   ========================================================= */

const startingBoard = [

    ["♜","♞","♝","♛","♚","♝","♞","♜"],

    ["♟","♟","♟","♟","♟","♟","♟","♟"],

    ["","","","","","","",""],

    ["","","","","","","",""],

    ["","","","","","","",""],

    ["","","","","","","",""],

    ["♙","♙","♙","♙","♙","♙","♙","♙"],

    ["♖","♘","♗","♕","♔","♗","♘","♖"]

];


/* =========================================================
   GAME STATE
   ========================================================= */

let board = copyBoard(startingBoard);

let whiteTurn = true;

let selectedRow = null;
let selectedCol = null;

let gameOver = false;


/* =========================================================
   CASTLING RIGHTS
   ========================================================= */

let whiteKingMoved = false;
let blackKingMoved = false;

let whiteRookAMoved = false;
let whiteRookHMoved = false;

let blackRookAMoved = false;
let blackRookHMoved = false;


/* =========================================================
   EN PASSANT
   ========================================================= */

let enPassantRow = null;
let enPassantCol = null;

let enPassantPawnRow = null;
let enPassantPawnCol = null;


/* =========================================================
   PROMOTION
   ========================================================= */

let promotionRow = null;
let promotionCol = null;
let promotionWhite = true;


/* =========================================================
   CHESS CLOCK
   ========================================================= */

let whiteTime = 10 * 60;
let blackTime = 10 * 60;

let clockInterval = null;


/* =========================================================
   COPY BOARD
   ========================================================= */

function copyBoard(b) {

    return b.map(row => row.slice());

}


/* =========================================================
   PIECE HELPERS
   ========================================================= */

function isWhitePiece(piece) {

    return "♔♕♖♗♘♙".includes(piece);

}


function isBlackPiece(piece) {

    return "♚♛♜♝♞♟".includes(piece);

}


function isWhite(piece) {

    return isWhitePiece(piece);

}


function sameColor(p1, p2) {

    if (p1 === "" || p2 === "") {
        return false;
    }

    return isWhite(p1) === isWhite(p2);

}


/* =========================================================
   BOARD DRAW
   ========================================================= */

function drawBoard() {

    const chessBoard =
        document.getElementById("chessBoard");

    chessBoard.innerHTML = "";

    for (let r = 0; r < 8; r++) {

        for (let c = 0; c < 8; c++) {

            const square = document.createElement("div");

            square.classList.add("square");

            if ((r + c) % 2 === 0) {
                square.classList.add("light");
            }
            else {
                square.classList.add("dark");
            }

            square.textContent = board[r][c];

            square.dataset.row = r;
            square.dataset.col = c;

            square.onclick = function() {

                squareClicked(r, c);

            };

            chessBoard.appendChild(square);

        }

    }

    highlightSelected();

}


/* =========================================================
   HIGHLIGHT
   ========================================================= */

function clearHighlights() {

    document
        .querySelectorAll(".square")
        .forEach(s => {

            s.classList.remove("selected");
            s.classList.remove("legal");
            s.classList.remove("capture");

        });

}


function highlightSelected() {

    clearHighlights();

    if (selectedRow === null) {
        return;
    }

    const selected =
        document.querySelector(
            `.square[data-row="${selectedRow}"][data-col="${selectedCol}"]`
        );

    if (selected) {
        selected.classList.add("selected");
    }

    const moves =
        getLegalMoves(selectedRow, selectedCol);

    moves.forEach(move => {

        const sq =
            document.querySelector(
                `.square[data-row="${move.row}"][data-col="${move.col}"]`
            );

        if (!sq) {
            return;
        }

        if (board[move.row][move.col] !== "") {
            sq.classList.add("capture");
        }
        else {
            sq.classList.add("legal");
        }

    });

}


/* =========================================================
   ATTACK DETECTION
   ========================================================= */

function squareAttacked(b, row, col, attackingWhite) {

    const pawn = attackingWhite ? "♙" : "♟";

    const pawnRow =
        attackingWhite ? row + 1 : row - 1;

    for (let dc of [-1, 1]) {

        const pc = col + dc;

        if (
            pawnRow >= 0 &&
            pawnRow < 8 &&
            pc >= 0 &&
            pc < 8 &&
            b[pawnRow][pc] === pawn
        ) {

            return true;

        }

    }


    const knight =
        attackingWhite ? "♘" : "♞";

    const knightMoves = [

        [-2,-1],
        [-2,1],
        [-1,-2],
        [-1,2],
        [1,-2],
        [1,2],
        [2,-1],
        [2,1]

    ];

    for (let [dr, dc] of knightMoves) {

        const r = row + dr;
        const c = col + dc;

        if (
            r >= 0 &&
            r < 8 &&
            c >= 0 &&
            c < 8 &&
            b[r][c] === knight
        ) {

            return true;

        }

    }


    const king =
        attackingWhite ? "♔" : "♚";

    for (let dr = -1; dr <= 1; dr++) {

        for (let dc = -1; dc <= 1; dc++) {

            if (dr === 0 && dc === 0) {
                continue;
            }

            const r = row + dr;
            const c = col + dc;

            if (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8 &&
                b[r][c] === king
            ) {

                return true;

            }

        }

    }


    const bishop =
        attackingWhite ? "♗" : "♝";

    const queen =
        attackingWhite ? "♕" : "♛";

    const rook =
        attackingWhite ? "♖" : "♜";


    const diagonalDirections = [

        [-1,-1],
        [-1,1],
        [1,-1],
        [1,1]

    ];


    for (let [dr, dc] of diagonalDirections) {

        let r = row + dr;
        let c = col + dc;

        while (
            r >= 0 &&
            r < 8 &&
            c >= 0 &&
            c < 8
        ) {

            const piece = b[r][c];

            if (piece !== "") {

                if (
                    piece === bishop ||
                    piece === queen
                ) {

                    return true;

                }

                break;

            }

            r += dr;
            c += dc;

        }

    }


    const straightDirections = [

        [-1,0],
        [1,0],
        [0,-1],
        [0,1]

    ];


    for (let [dr, dc] of straightDirections) {

        let r = row + dr;
        let c = col + dc;

        while (
            r >= 0 &&
            r < 8 &&
            c >= 0 &&
            c < 8
        ) {

            const piece = b[r][c];

            if (piece !== "") {

                if (
                    piece === rook ||
                    piece === queen
                ) {

                    return true;

                }

                break;

            }

            r += dr;
            c += dc;

        }

    }


    return false;

}


/* =========================================================
   CHECK
   ========================================================= */

function isInCheck(b, white) {

    const king =
        white ? "♔" : "♚";

    let kr = -1;
    let kc = -1;


    for (let r = 0; r < 8; r++) {

        for (let c = 0; c < 8; c++) {

            if (b[r][c] === king) {

                kr = r;
                kc = c;

            }

        }

    }


    if (kr === -1) {
        return true;
    }


    return squareAttacked(
        b,
        kr,
        kc,
        !white
    );

}


/* =========================================================
   PSEUDO MOVES
   ========================================================= */

function getPseudoMoves(
    b,
    row,
    col,
    epRow,
    epCol
) {

    const piece = b[row][col];

    const moves = [];

    if (piece === "") {
        return moves;
    }


    const white = isWhite(piece);


    function add(r, c) {

        if (
            r < 0 ||
            r >= 8 ||
            c < 0 ||
            c >= 8
        ) {
            return;
        }

        if (
            b[r][c] !== "" &&
            sameColor(piece, b[r][c])
        ) {
            return;
        }

        if (
            b[r][c] === "♔" ||
            b[r][c] === "♚"
        ) {
            return;
        }

        moves.push({
            row: r,
            col: c
        });

    }


    /* PAWN */

    if (piece === "♙" || piece === "♟") {

        const dir = white ? -1 : 1;

        const startRow = white ? 6 : 1;

        const oneRow = row + dir;

        if (
            oneRow >= 0 &&
            oneRow < 8 &&
            b[oneRow][col] === ""
        ) {

            moves.push({
                row: oneRow,
                col: col
            });


            const twoRow = row + 2 * dir;

            if (
                row === startRow &&
                b[twoRow][col] === ""
            ) {

                moves.push({
                    row: twoRow,
                    col: col
                });

            }

        }


        for (let dc of [-1, 1]) {

            const r = row + dir;
            const c = col + dc;

            if (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8
            ) {

                if (
                    b[r][c] !== "" &&
                    !sameColor(piece, b[r][c]) &&
                    b[r][c] !== "♔" &&
                    b[r][c] !== "♚"
                ) {

                    moves.push({
                        row: r,
                        col: c
                    });

                }


                if (
                    r === epRow &&
                    c === epCol &&
                    b[r][c] === ""
                ) {

                    moves.push({
                        row: r,
                        col: c,
                        enPassant: true
                    });

                }

            }

        }

    }


    /* KNIGHT */

    else if (
        piece === "♘" ||
        piece === "♞"
    ) {

        const jumps = [

            [-2,-1],
            [-2,1],
            [-1,-2],
            [-1,2],
            [1,-2],
            [1,2],
            [2,-1],
            [2,1]

        ];

        jumps.forEach(([dr, dc]) => {

            add(row + dr, col + dc);

        });

    }


    /* KING */

    else if (
        piece === "♔" ||
        piece === "♚"
    ) {

        for (let dr = -1; dr <= 1; dr++) {

            for (let dc = -1; dc <= 1; dc++) {

                if (dr !== 0 || dc !== 0) {

                    add(
                        row + dr,
                        col + dc
                    );

                }

            }

        }

    }


    /* BISHOP */

    else if (
        piece === "♗" ||
        piece === "♝"
    ) {

        const directions = [

            [-1,-1],
            [-1,1],
            [1,-1],
            [1,1]

        ];

        for (let [dr, dc] of directions) {

            let r = row + dr;
            let c = col + dc;

            while (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8
            ) {

                if (b[r][c] === "") {

                    moves.push({
                        row: r,
                        col: c
                    });

                }
                else {

                    if (
                        !sameColor(
                            piece,
                            b[r][c]
                        ) &&
                        b[r][c] !== "♔" &&
                        b[r][c] !== "♚"
                    ) {

                        moves.push({
                            row: r,
                            col: c
                        });

                    }

                    break;

                }

                r += dr;
                c += dc;

            }

        }

    }


    /* ROOK */

    else if (
        piece === "♖" ||
        piece === "♜"
    ) {

        const directions = [

            [-1,0],
            [1,0],
            [0,-1],
            [0,1]

        ];

        for (let [dr, dc] of directions) {

            let r = row + dr;
            let c = col + dc;

            while (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8
            ) {

                if (b[r][c] === "") {

                    moves.push({
                        row: r,
                        col: c
                    });

                }
                else {

                    if (
                        !sameColor(
                            piece,
                            b[r][c]
                        ) &&
                        b[r][c] !== "♔" &&
                        b[r][c] !== "♚"
                    ) {

                        moves.push({
                            row: r,
                            col: c
                        });

                    }

                    break;

                }

                r += dr;
                c += dc;

            }

        }

    }


    /* QUEEN */

    else if (
        piece === "♕" ||
        piece === "♛"
    ) {

        const directions = [

            [-1,-1],
            [-1,1],
            [1,-1],
            [1,1],

            [-1,0],
            [1,0],
            [0,-1],
            [0,1]

        ];

        for (let [dr, dc] of directions) {

            let r = row + dr;
            let c = col + dc;

            while (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8
            ) {

                if (b[r][c] === "") {

                    moves.push({
                        row: r,
                        col: c
                    });

                }
                else {

                    if (
                        !sameColor(
                            piece,
                            b[r][c]
                        ) &&
                        b[r][c] !== "♔" &&
                        b[r][c] !== "♚"
                    ) {

                        moves.push({
                            row: r,
                            col: c
                        });

                    }

                    break;

                }

                r += dr;
                c += dc;

            }

        }

    }


    return moves;

}


/* =========================================================
   CASTLING MOVES
   ========================================================= */

function getCastlingMoves(b, row, col) {

    const moves = [];

    const piece = b[row][col];

    if (piece !== "♔" && piece !== "♚") {
        return moves;
    }


    const white = piece === "♔";


    if (white) {

        if (
            row === 7 &&
            col === 4 &&
            !whiteKingMoved &&
            !whiteRookHMoved &&
            b[7][7] === "♖" &&
            b[7][5] === "" &&
            b[7][6] === "" &&
            !squareAttacked(b,7,4,false) &&
            !squareAttacked(b,7,5,false) &&
            !squareAttacked(b,7,6,false)
        ) {

            moves.push({
                row: 7,
                col: 6,
                castle: "kingside"
            });

        }


        if (
            row === 7 &&
            col === 4 &&
            !whiteKingMoved &&
            !whiteRookAMoved &&
            b[7][0] === "♖" &&
            b[7][1] === "" &&
            b[7][2] === "" &&
            b[7][3] === "" &&
            !squareAttacked(b,7,4,false) &&
            !squareAttacked(b,7,3,false) &&
            !squareAttacked(b,7,2,false)
        ) {

            moves.push({
                row: 7,
                col: 2,
                castle: "queenside"
            });

        }

    }
    else {

        if (
            row === 0 &&
            col === 4 &&
            !blackKingMoved &&
            !blackRookHMoved &&
            b[0][7] === "♜" &&
            b[0][5] === "" &&
            b[0][6] === "" &&
            !squareAttacked(b,0,4,true) &&
            !squareAttacked(b,0,5,true) &&
            !squareAttacked(b,0,6,true)
        ) {

            moves.push({
                row: 0,
                col: 6,
                castle: "kingside"
            });

        }


        if (
            row === 0 &&
            col === 4 &&
            !blackKingMoved &&
            !blackRookAMoved &&
            b[0][0] === "♜" &&
            b[0][1] === "" &&
            b[0][2] === "" &&
            b[0][3] === "" &&
            !squareAttacked(b,0,4,true) &&
            !squareAttacked(b,0,3,true) &&
            !squareAttacked(b,0,2,true)
        ) {

            moves.push({
                row: 0,
                col: 2,
                castle: "queenside"
            });

        }

    }


    return moves;

}


/* =========================================================
   SIMULATE MOVE
   ========================================================= */

function simulateMove(
    b,
    fr,
    fc,
    tr,
    tc,
    epR,
    epC,
    epPawnR,
    epPawnC
) {

    const copy = copyBoard(b);

    const piece = copy[fr][fc];

    copy[fr][fc] = "";

    /* EN PASSANT */

    if (
        piece === "♙" ||
        piece === "♟"
    ) {

        if (
            tr === epR &&
            tc === epC &&
            copy[tr][tc] === "" &&
            epPawnR !== null
        ) {

            copy[epPawnR][epPawnC] = "";

        }

    }


    copy[tr][tc] = piece;


    /* CASTLING */

    if (
        (piece === "♔" || piece === "♚") &&
        Math.abs(tc - fc) === 2
    ) {

        if (tc === 6) {

            copy[tr][5] = copy[tr][7];
            copy[tr][7] = "";

        }
        else if (tc === 2) {

            copy[tr][3] = copy[tr][0];
            copy[tr][0] = "";

        }

    }


    return copy;

}


/* =========================================================
   LEGAL MOVES
   ========================================================= */

function getLegalMoves(row, col) {

    const piece = board[row][col];

    if (piece === "") {
        return [];
    }


    const white = isWhite(piece);


    const pseudo =
        getPseudoMoves(
            board,
            row,
            col,
            enPassantRow,
            enPassantCol
        );


    const castle =
        getCastlingMoves(
            board,
            row,
            col
        );


    const allMoves =
        pseudo.concat(castle);


    const legal = [];


    for (let move of allMoves) {

        const testBoard =
            simulateMove(
                board,
                row,
                col,
                move.row,
                move.col,
                enPassantRow,
                enPassantCol,
                enPassantPawnRow,
                enPassantPawnCol
            );


        if (
            !isInCheck(
                testBoard,
                white
            )
        ) {

            legal.push(move);

        }

    }


    return legal;

}


/* =========================================================
   MAKE MOVE
   ========================================================= */

function makeMove(
    fr,
    fc,
    tr,
    tc,
    move
) {

    const piece = board[fr][fc];


    /* EN PASSANT */

    if (move.enPassant) {

        if (enPassantPawnRow !== null) {

            board[enPassantPawnRow]
                 [enPassantPawnCol] = "";

        }

    }


    board[fr][fc] = "";

    board[tr][tc] = piece;


    /* CASTLING */

    if (
        (piece === "♔" || piece === "♚") &&
        Math.abs(tc - fc) === 2
    ) {

        if (tc === 6) {

            board[tr][5] = board[tr][7];
            board[tr][7] = "";

        }
        else {

            board[tr][3] = board[tr][0];
            board[tr][0] = "";

        }

    }


    /* KING MOVED */

    if (piece === "♔") {
        whiteKingMoved = true;
    }

    if (piece === "♚") {
        blackKingMoved = true;
    }


    /* ROOK MOVED */

    if (piece === "♖") {

        if (fr === 7 && fc === 0) {
            whiteRookAMoved = true;
        }

        if (fr === 7 && fc === 7) {
            whiteRookHMoved = true;
        }

    }


    if (piece === "♜") {

        if (fr === 0 && fc === 0) {
            blackRookAMoved = true;
        }

        if (fr === 0 && fc === 7) {
            blackRookHMoved = true;
        }

    }


    /* ROOK CAPTURED */

    if (tr === 7 && tc === 0) {
        whiteRookAMoved = true;
    }

    if (tr === 7 && tc === 7) {
        whiteRookHMoved = true;
    }

    if (tr === 0 && tc === 0) {
        blackRookAMoved = true;
    }

    if (tr === 0 && tc === 7) {
        blackRookHMoved = true;
    }


    /* RESET EN PASSANT */

    enPassantRow = null;
    enPassantCol = null;
    enPassantPawnRow = null;
    enPassantPawnCol = null;


    /* CREATE EN PASSANT */

    if (
        piece === "♙" &&
        fr === 6 &&
        tr === 4
    ) {

        enPassantRow = 5;
        enPassantCol = fc;

        enPassantPawnRow = tr;
        enPassantPawnCol = tc;

    }


    if (
        piece === "♟" &&
        fr === 1 &&
        tr === 3
    ) {

        enPassantRow = 2;
        enPassantCol = fc;

        enPassantPawnRow = tr;
        enPassantPawnCol = tc;

    }


    /* PROMOTION */

    if (
        piece === "♙" &&
        tr === 0
    ) {

        promotionRow = tr;
        promotionCol = tc;
        promotionWhite = true;

        document.getElementById(
            "promotionMenu"
        ).style.display = "block";

        document.getElementById(
            "promotionPlayer"
        ).textContent =
            "Choose White Promotion";

        return true;

    }


    if (
        piece === "♟" &&
        tr === 7
    ) {

        promotionRow = tr;
        promotionCol = tc;
        promotionWhite = false;

        document.getElementById(
            "promotionMenu"
        ).style.display = "block";

        document.getElementById(
            "promotionPlayer"
        ).textContent =
            "Choose Black Promotion";

        return true;

    }


    return false;

}


/* =========================================================
   PROMOTION
   ========================================================= */

function finishPromotion(type) {

    const piecesWhite = {

        queen: "♕",
        rook: "♖",
        bishop: "♗",
        knight: "♘"

    };


    const piecesBlack = {

        queen: "♛",
        rook: "♜",
        bishop: "♝",
        knight: "♞"

    };


    if (promotionWhite) {

        board[promotionRow][promotionCol] =
            piecesWhite[type];

    }
    else {

        board[promotionRow][promotionCol] =
            piecesBlack[type];

    }


    document.getElementById(
        "promotionMenu"
    ).style.display = "none";


    promotionRow = null;
    promotionCol = null;


    whiteTurn = !whiteTurn;

    selectedRow = null;
    selectedCol = null;


    drawBoard();

    updateTurnText();

    checkGameStatus();

}


/* =========================================================
   ANY LEGAL MOVE?
   ========================================================= */

function hasAnyLegalMove(white) {

    for (let r = 0; r < 8; r++) {

        for (let c = 0; c < 8; c++) {

            const piece = board[r][c];

            if (
                piece !== "" &&
                isWhite(piece) === white
            ) {

                if (
                    getLegalMoves(r,c).length > 0
                ) {

                    return true;

                }

            }

        }

    }


    return false;

}


/* =========================================================
   CHECK / CHECKMATE / STALEMATE
   ========================================================= */

function checkGameStatus() {

    const currentWhite = whiteTurn;

    const check =
        isInCheck(
            board,
            currentWhite
        );


    const hasMove =
        hasAnyLegalMove(
            currentWhite
        );


    const status =
        document.getElementById(
            "statusText"
        );


    const message =
        document.getElementById(
            "messageText"
        );


    if (!hasMove && check) {

        gameOver = true;

        if (currentWhite) {

            status.textContent =
                "☠️ Checkmate! Black wins!";

        }
        else {

            status.textContent =
                "☠️ Checkmate! White wins!";

        }

        message.textContent = "";

        stopClock();

        return;

    }


    if (!hasMove && !check) {

        gameOver = true;

        status.textContent =
            "🤝 Stalemate! Draw!";

        message.textContent = "";

        stopClock();

        return;

    }


    if (check) {

        status.textContent =
            "⚠️ CHECK!";

    }
    else {

        status.textContent = "";

    }

}


/* =========================================================
   CLICK HANDLER
   ========================================================= */

function squareClicked(row, col) {

    if (gameOver) {
        return;
    }


    if (
        promotionRow !== null
    ) {
        return;
    }


    const piece = board[row][col];


    /* NOTHING SELECTED */

    if (
        selectedRow === null
    ) {

        if (
            piece !== "" &&
            isWhite(piece) === whiteTurn
        ) {

            selectedRow = row;
            selectedCol = col;

            highlightSelected();

        }

        return;

    }


    /* SAME SQUARE = DESELECT */

    if (
        selectedRow === row &&
        selectedCol === col
    ) {

        selectedRow = null;
        selectedCol = null;

        highlightSelected();

        return;

    }


    /* SELECT ANOTHER PIECE OF SAME COLOR */

    if (
        piece !== "" &&
        isWhite(piece) === whiteTurn
    ) {

        selectedRow = row;
        selectedCol = col;

        highlightSelected();

        return;

    }


    /* CHECK LEGAL MOVE */

    const moves =
        getLegalMoves(
            selectedRow,
            selectedCol
        );


    const move =
        moves.find(
            m =>
                m.row === row &&
                m.col === col
        );


    if (!move) {

        document.getElementById(
            "messageText"
        ).textContent =
            "❌ Illegal move";

        return;

    }


    document.getElementById(
        "messageText"
    ).textContent = "";


    const wasPromotion =
        makeMove(
            selectedRow,
            selectedCol,
            row,
            col,
            move
        );


    selectedRow = null;
    selectedCol = null;


    if (wasPromotion) {

        drawBoard();

        return;

    }


    whiteTurn = !whiteTurn;

    drawBoard();

    updateTurnText();

    checkGameStatus();

}


/* =========================================================
   TURN TEXT
   ========================================================= */

function updateTurnText() {

    document.getElementById(
        "turnText"
    ).textContent =
        whiteTurn
        ? "White's Turn"
        : "Black's Turn";

}


/* =========================================================
   CHESS CLOCK
   ========================================================= */

function formatTime(seconds) {

    const minutes =
        Math.floor(seconds / 60);

    const secs =
        seconds % 60;


    return (
        String(minutes).padStart(2,"0")
        +
        ":"
        +
        String(secs).padStart(2,"0")
    );

}


function updateClocks() {

    document.getElementById(
        "whiteClock"
    ).textContent =
        formatTime(whiteTime);


    document.getElementById(
        "blackClock"
    ).textContent =
        formatTime(blackTime);

}


function stopClock() {

    if (clockInterval !== null) {

        clearInterval(clockInterval);

        clockInterval = null;

    }

}


function startClock() {

    stopClock();


    clockInterval =
        setInterval(function() {

            if (gameOver) {

                stopClock();

                return;

            }


            if (whiteTurn) {

                whiteTime--;

            }
            else {

                blackTime--;

            }


            updateClocks();


            if (whiteTime <= 0) {

                whiteTime = 0;

                gameOver = true;

                stopClock();

                document.getElementById(
                    "statusText"
                ).textContent =
                    "⏱️ Time's up! Black wins!";

                return;

            }


            if (blackTime <= 0) {

                blackTime = 0;

                gameOver = true;

                stopClock();

                document.getElementById(
                    "statusText"
                ).textContent =
                    "⏱️ Time's up! White wins!";

                return;

            }

        }, 1000);

}


/* =========================================================
   RESTART GAME
   ========================================================= */

function restartGame() {

    board =
        copyBoard(startingBoard);


    whiteTurn = true;

    selectedRow = null;
    selectedCol = null;

    gameOver = false;


    whiteKingMoved = false;
    blackKingMoved = false;

    whiteRookAMoved = false;
    whiteRookHMoved = false;

    blackRookAMoved = false;
    blackRookHMoved = false;


    enPassantRow = null;
    enPassantCol = null;

    enPassantPawnRow = null;
    enPassantPawnCol = null;


    promotionRow = null;
    promotionCol = null;


    whiteTime = 10 * 60;
    blackTime = 10 * 60;


    document.getElementById(
        "promotionMenu"
    ).style.display = "none";


    document.getElementById(
        "messageText"
    ).textContent = "";


    document.getElementById(
        "statusText"
    ).textContent = "";


    updateTurnText();

    updateClocks();

    drawBoard();

    startClock();

}


/* =========================================================
   START GAME
   ========================================================= */

updateTurnText();

updateClocks();

drawBoard();

startClock();

</script>

</body>
</html>
"""

display(HTML(html))

In [ ]:
from IPython.display import HTML, display

html = r"""
<!DOCTYPE html>
<html>
<head>

<style>

#chessGame {
    font-family: Arial, sans-serif;
    text-align: center;
    margin-top: 20px;
}

#chessBoard {
    width: 480px;
    height: 480px;
    margin: 20px auto;
    display: grid;
    grid-template-columns: repeat(8, 1fr);
    border: 4px solid #222;
}

.square {
    width: 60px;
    height: 60px;
    display: flex;
    justify-content: center;
    align-items: center;
    font-size: 42px;
    cursor: pointer;
    user-select: none;
    box-sizing: border-box;
}

.light {
    background: #f0d9b5;
}

.dark {
    background: #b58863;
}

.selected {
    outline: 4px solid #00aaff;
    outline-offset: -4px;
}

.legal {
    box-shadow: inset 0 0 0 5px rgba(0,180,0,0.35);
}

.capture {
    box-shadow: inset 0 0 0 5px rgba(220,0,0,0.45);
}

#turnText {
    font-size: 22px;
    font-weight: bold;
    margin: 10px;
}

#statusText {
    font-size: 20px;
    font-weight: bold;
    margin: 10px;
}

#messageText {
    font-size: 18px;
    margin: 10px;
}

.clocks {
    display: flex;
    justify-content: center;
    gap: 30px;
    margin: 15px;
}

.clockBox {
    background: #222;
    color: white;
    padding: 10px 25px;
    border-radius: 10px;
    min-width: 120px;
}

.clockLabel {
    font-size: 16px;
}

.clockTime {
    font-size: 30px;
    font-weight: bold;
    margin-top: 5px;
}

#restartButton {
    padding: 10px 20px;
    font-size: 16px;
    border: none;
    border-radius: 8px;
    background: #333;
    color: white;
    cursor: pointer;
}

#restartButton:hover {
    background: #555;
}

#promotionMenu {
    display: none;
    position: fixed;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    background: white;
    border: 3px solid #222;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 5px 25px rgba(0,0,0,0.4);
    z-index: 1000;
}

#promotionMenu h3 {
    margin-top: 0;
}

.promotionButton {
    font-size: 40px;
    padding: 10px 15px;
    margin: 5px;
    cursor: pointer;
    border: 2px solid #aaa;
    border-radius: 8px;
    background: #f5f5f5;
}

.promotionButton:hover {
    background: #ddd;
}

.aiInfo {
    font-size: 16px;
    margin: 8px;
}

</style>

</head>

<body>

<div id="chessGame">

    <h1>♟️ Chess vs Computer 🤖</h1>

    <div class="aiInfo">
        You are White • Computer is Black
    </div>

    <div class="clocks">

        <div class="clockBox">
            <div class="clockLabel">♔ You</div>
            <div id="whiteClock" class="clockTime">10:00</div>
        </div>

        <div class="clockBox">
            <div class="clockLabel">♚ Computer</div>
            <div id="blackClock" class="clockTime">10:00</div>
        </div>

    </div>

    <div id="turnText">Your Turn</div>

    <div id="statusText"></div>

    <div id="messageText"></div>

    <div id="chessBoard"></div>

    <button id="restartButton" onclick="restartGame()">
        🔄 Restart Game
    </button>

</div>


<div id="promotionMenu">

    <h3 id="promotionPlayer">Choose Promotion</h3>

    <button class="promotionButton" onclick="finishPromotion('queen')">
        ♕
    </button>

    <button class="promotionButton" onclick="finishPromotion('rook')">
        ♖
    </button>

    <button class="promotionButton" onclick="finishPromotion('bishop')">
        ♗
    </button>

    <button class="promotionButton" onclick="finishPromotion('knight')">
        ♘
    </button>

</div>


<script>

/* =========================================================
   STARTING BOARD
   ========================================================= */

const startingBoard = [

    ["♜","♞","♝","♛","♚","♝","♞","♜"],

    ["♟","♟","♟","♟","♟","♟","♟","♟"],

    ["","","","","","","",""],

    ["","","","","","","",""],

    ["","","","","","","",""],

    ["","","","","","","",""],

    ["♙","♙","♙","♙","♙","♙","♙","♙"],

    ["♖","♘","♗","♕","♔","♗","♘","♖"]

];


/* =========================================================
   GAME STATE
   ========================================================= */

let board = copyBoard(startingBoard);

let whiteTurn = true;

let selectedRow = null;
let selectedCol = null;

let gameOver = false;

let computerThinking = false;


/* =========================================================
   CASTLING
   ========================================================= */

let whiteKingMoved = false;
let blackKingMoved = false;

let whiteRookAMoved = false;
let whiteRookHMoved = false;

let blackRookAMoved = false;
let blackRookHMoved = false;


/* =========================================================
   EN PASSANT
   ========================================================= */

let enPassantRow = null;
let enPassantCol = null;

let enPassantPawnRow = null;
let enPassantPawnCol = null;


/* =========================================================
   PROMOTION
   ========================================================= */

let promotionRow = null;
let promotionCol = null;
let promotionWhite = true;


/* =========================================================
   CLOCK
   ========================================================= */

let whiteTime = 10 * 60;
let blackTime = 10 * 60;

let clockInterval = null;


/* =========================================================
   COPY BOARD
   ========================================================= */

function copyBoard(b) {

    return b.map(row => row.slice());

}


/* =========================================================
   PIECE HELPERS
   ========================================================= */

function isWhitePiece(piece) {

    return "♔♕♖♗♘♙".includes(piece);

}


function isBlackPiece(piece) {

    return "♚♛♜♝♞♟".includes(piece);

}


function isWhite(piece) {

    return isWhitePiece(piece);

}


function sameColor(p1, p2) {

    if (p1 === "" || p2 === "") {
        return false;
    }

    return isWhite(p1) === isWhite(p2);

}


/* =========================================================
   DRAW BOARD
   ========================================================= */

function drawBoard() {

    const chessBoard =
        document.getElementById("chessBoard");

    chessBoard.innerHTML = "";

    for (let r = 0; r < 8; r++) {

        for (let c = 0; c < 8; c++) {

            const square =
                document.createElement("div");

            square.classList.add("square");

            if ((r + c) % 2 === 0) {
                square.classList.add("light");
            }
            else {
                square.classList.add("dark");
            }

            square.textContent = board[r][c];

            square.dataset.row = r;
            square.dataset.col = c;

            square.onclick = function() {

                squareClicked(r,c);

            };

            chessBoard.appendChild(square);

        }

    }

    highlightSelected();

}


/* =========================================================
   HIGHLIGHT
   ========================================================= */

function clearHighlights() {

    document
        .querySelectorAll(".square")
        .forEach(s => {

            s.classList.remove("selected");
            s.classList.remove("legal");
            s.classList.remove("capture");

        });

}


function highlightSelected() {

    clearHighlights();

    if (selectedRow === null) {
        return;
    }

    const selected =
        document.querySelector(
            `.square[data-row="${selectedRow}"][data-col="${selectedCol}"]`
        );

    if (selected) {
        selected.classList.add("selected");
    }


    const moves =
        getLegalMoves(
            selectedRow,
            selectedCol
        );


    moves.forEach(move => {

        const sq =
            document.querySelector(
                `.square[data-row="${move.row}"][data-col="${move.col}"]`
            );

        if (!sq) {
            return;
        }

        if (board[move.row][move.col] !== "") {
            sq.classList.add("capture");
        }
        else {
            sq.classList.add("legal");
        }

    });

}


/* =========================================================
   ATTACK DETECTION
   ========================================================= */

function squareAttacked(
    b,
    row,
    col,
    attackingWhite
) {

    const pawn =
        attackingWhite ? "♙" : "♟";

    const pawnRow =
        attackingWhite ? row + 1 : row - 1;


    for (let dc of [-1,1]) {

        const pc = col + dc;

        if (
            pawnRow >= 0 &&
            pawnRow < 8 &&
            pc >= 0 &&
            pc < 8 &&
            b[pawnRow][pc] === pawn
        ) {

            return true;

        }

    }


    const knight =
        attackingWhite ? "♘" : "♞";


    const knightMoves = [

        [-2,-1],
        [-2,1],
        [-1,-2],
        [-1,2],
        [1,-2],
        [1,2],
        [2,-1],
        [2,1]

    ];


    for (let [dr,dc] of knightMoves) {

        const r = row + dr;
        const c = col + dc;

        if (
            r >= 0 &&
            r < 8 &&
            c >= 0 &&
            c < 8 &&
            b[r][c] === knight
        ) {

            return true;

        }

    }


    const king =
        attackingWhite ? "♔" : "♚";


    for (let dr=-1; dr<=1; dr++) {

        for (let dc=-1; dc<=1; dc++) {

            if (dr === 0 && dc === 0) {
                continue;
            }

            const r = row + dr;
            const c = col + dc;

            if (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8 &&
                b[r][c] === king
            ) {

                return true;

            }

        }

    }


    const bishop =
        attackingWhite ? "♗" : "♝";

    const queen =
        attackingWhite ? "♕" : "♛";

    const rook =
        attackingWhite ? "♖" : "♜";


    const diagonals = [

        [-1,-1],
        [-1,1],
        [1,-1],
        [1,1]

    ];


    for (let [dr,dc] of diagonals) {

        let r = row + dr;
        let c = col + dc;

        while (
            r >= 0 &&
            r < 8 &&
            c >= 0 &&
            c < 8
        ) {

            const piece = b[r][c];

            if (piece !== "") {

                if (
                    piece === bishop ||
                    piece === queen
                ) {

                    return true;

                }

                break;

            }

            r += dr;
            c += dc;

        }

    }


    const straight = [

        [-1,0],
        [1,0],
        [0,-1],
        [0,1]

    ];


    for (let [dr,dc] of straight) {

        let r = row + dr;
        let c = col + dc;

        while (
            r >= 0 &&
            r < 8 &&
            c >= 0 &&
            c < 8
        ) {

            const piece = b[r][c];

            if (piece !== "") {

                if (
                    piece === rook ||
                    piece === queen
                ) {

                    return true;

                }

                break;

            }

            r += dr;
            c += dc;

        }

    }


    return false;

}


/* =========================================================
   CHECK
   ========================================================= */

function isInCheck(b, white) {

    const king =
        white ? "♔" : "♚";


    let kr = -1;
    let kc = -1;


    for (let r=0; r<8; r++) {

        for (let c=0; c<8; c++) {

            if (b[r][c] === king) {

                kr = r;
                kc = c;

            }

        }

    }


    if (kr === -1) {
        return true;
    }


    return squareAttacked(
        b,
        kr,
        kc,
        !white
    );

}


/* =========================================================
   PSEUDO MOVES
   ========================================================= */

function getPseudoMoves(
    b,
    row,
    col,
    epRow,
    epCol
) {

    const piece = b[row][col];

    const moves = [];

    if (piece === "") {
        return moves;
    }


    const white = isWhite(piece);


    function add(r,c) {

        if (
            r < 0 ||
            r >= 8 ||
            c < 0 ||
            c >= 8
        ) {
            return;
        }


        if (
            b[r][c] !== "" &&
            sameColor(piece,b[r][c])
        ) {
            return;
        }


        if (
            b[r][c] === "♔" ||
            b[r][c] === "♚"
        ) {
            return;
        }


        moves.push({
            row:r,
            col:c
        });

    }


    /* PAWN */

    if (
        piece === "♙" ||
        piece === "♟"
    ) {

        const dir =
            white ? -1 : 1;

        const startRow =
            white ? 6 : 1;

        const oneRow =
            row + dir;


        if (
            oneRow >= 0 &&
            oneRow < 8 &&
            b[oneRow][col] === ""
        ) {

            moves.push({
                row:oneRow,
                col:col
            });


            const twoRow =
                row + 2 * dir;


            if (
                row === startRow &&
                b[twoRow][col] === ""
            ) {

                moves.push({
                    row:twoRow,
                    col:col
                });

            }

        }


        for (let dc of [-1,1]) {

            const r = row + dir;
            const c = col + dc;


            if (
                r >= 0 &&
                r < 8 &&
                c >= 0 &&
                c < 8
            ) {

                if (
                    b[r][c] !== "" &&
                    !sameColor(
                        piece,
                        b[r][c]
                    ) &&
                    b[r][c] !== "♔" &&
                    b[r][c] !== "♚"
                ) {

                    moves.push({
                        row:r,
                        col:c
                    });

                }


                if (
                    r === epRow &&
                    c === epCol &&
                    b[r][c] === ""
                ) {

                    moves.push({
                        row:r,
                        col:c,
                        enPassant:true
                    });

                }

            }

        }

    }


    /* KNIGHT */

    else if (
        piece === "♘" ||
        piece === "♞"
    ) {

        const jumps = [

            [-2,-1],
            [-2,1],
            [-1,-2],
            [-1,2],
            [1,-2],
            [1,2],
            [2,-1],
            [2,1]

        ];


        jumps.forEach(
            ([dr,dc]) =>
                add(
                    row+dr,
                    col+dc
                )
        );

    }


    /* KING */

    else if (
        piece === "♔" ||
        piece === "♚"
    ) {

        for (let dr=-1; dr<=1; dr++) {

            for (let dc=-1; dc<=1; dc++) {

                if (
                    dr !== 0 ||
                    dc !== 0
                ) {

                    add(
                        row+dr,
                        col+dc
                    );

                }

            }

        }

    }


    /* BISHOP */

    else if (
        piece === "♗" ||
        piece === "♝"
    ) {

        const dirs = [

            [-1,-1],
            [-1,1],
            [1,-1],
            [1,1]

        ];


        for (let [dr,dc] of dirs) {

            let r = row+dr;
            let c = col+dc;


            while (
                r>=0 &&
                r<8 &&
                c>=0 &&
                c<8
            ) {

                if (b[r][c] === "") {

                    moves.push({
                        row:r,
                        col:c
                    });

                }
                else {

                    if (
                        !sameColor(
                            piece,
                            b[r][c]
                        ) &&
                        b[r][c] !== "♔" &&
                        b[r][c] !== "♚"
                    ) {

                        moves.push({
                            row:r,
                            col:c
                        });

                    }

                    break;

                }


                r += dr;
                c += dc;

            }

        }

    }


    /* ROOK */

    else if (
        piece === "♖" ||
        piece === "♜"
    ) {

        const dirs = [

            [-1,0],
            [1,0],
            [0,-1],
            [0,1]

        ];


        for (let [dr,dc] of dirs) {

            let r = row+dr;
            let c = col+dc;


            while (
                r>=0 &&
                r<8 &&
                c>=0 &&
                c<8
            ) {

                if (b[r][c] === "") {

                    moves.push({
                        row:r,
                        col:c
                    });

                }
                else {

                    if (
                        !sameColor(
                            piece,
                            b[r][c]
                        ) &&
                        b[r][c] !== "♔" &&
                        b[r][c] !== "♚"
                    ) {

                        moves.push({
                            row:r,
                            col:c
                        });

                    }

                    break;

                }


                r += dr;
                c += dc;

            }

        }

    }


    /* QUEEN */

    else if (
        piece === "♕" ||
        piece === "♛"
    ) {

        const dirs = [

            [-1,-1],
            [-1,1],
            [1,-1],
            [1,1],

            [-1,0],
            [1,0],
            [0,-1],
            [0,1]

        ];


        for (let [dr,dc] of dirs) {

            let r = row+dr;
            let c = col+dc;


            while (
                r>=0 &&
                r<8 &&
                c>=0 &&
                c<8
            ) {

                if (b[r][c] === "") {

                    moves.push({
                        row:r,
                        col:c
                    });

                }
                else {

                    if (
                        !sameColor(
                            piece,
                            b[r][c]
                        ) &&
                        b[r][c] !== "♔" &&
                        b[r][c] !== "♚"
                    ) {

                        moves.push({
                            row:r,
                            col:c
                        });

                    }

                    break;

                }


                r += dr;
                c += dc;

            }

        }

    }


    return moves;

}


/* =========================================================
   CASTLING
   ========================================================= */

function getCastlingMoves(
    b,
    row,
    col
) {

    const moves = [];

    const piece = b[row][col];


    if (
        piece !== "♔" &&
        piece !== "♚"
    ) {
        return moves;
    }


    if (piece === "♔") {

        if (
            row === 7 &&
            col === 4 &&
            !whiteKingMoved &&
            !whiteRookHMoved &&
            b[7][7] === "♖" &&
            b[7][5] === "" &&
            b[7][6] === "" &&
            !squareAttacked(b,7,4,false) &&
            !squareAttacked(b,7,5,false) &&
            !squareAttacked(b,7,6,false)
        ) {

            moves.push({
                row:7,
                col:6,
                castle:"kingside"
            });

        }


        if (
            row === 7 &&
            col === 4 &&
            !whiteKingMoved &&
            !whiteRookAMoved &&
            b[7][0] === "♖" &&
            b[7][1] === "" &&
            b[7][2] === "" &&
            b[7][3] === "" &&
            !squareAttacked(b,7,4,false) &&
            !squareAttacked(b,7,3,false) &&
            !squareAttacked(b,7,2,false)
        ) {

            moves.push({
                row:7,
                col:2,
                castle:"queenside"
            });

        }

    }
    else {

        if (
            row === 0 &&
            col === 4 &&
            !blackKingMoved &&
            !blackRookHMoved &&
            b[0][7] === "♜" &&
            b[0][5] === "" &&
            b[0][6] === "" &&
            !squareAttacked(b,0,4,true) &&
            !squareAttacked(b,0,5,true) &&
            !squareAttacked(b,0,6,true)
        ) {

            moves.push({
                row:0,
                col:6,
                castle:"kingside"
            });

        }


        if (
            row === 0 &&
            col === 4 &&
            !blackKingMoved &&
            !blackRookAMoved &&
            b[0][0] === "♜" &&
            b[0][1] === "" &&
            b[0][2] === "" &&
            b[0][3] === "" &&
            !squareAttacked(b,0,4,true) &&
            !squareAttacked(b,0,3,true) &&
            !squareAttacked(b,0,2,true)
        ) {

            moves.push({
                row:0,
                col:2,
                castle:"queenside"
            });

        }

    }


    return moves;

}


/* =========================================================
   SIMULATE MOVE
   ========================================================= */

function simulateMove(
    b,
    fr,
    fc,
    tr,
    tc,
    epR,
    epC,
    epPawnR,
    epPawnC
) {

    const copy = copyBoard(b);

    const piece = copy[fr][fc];

    copy[fr][fc] = "";


    /* EN PASSANT */

    if (
        (piece === "♙" || piece === "♟") &&
        tr === epR &&
        tc === epC &&
        copy[tr][tc] === "" &&
        epPawnR !== null
    ) {

        copy[epPawnR][epPawnC] = "";

    }


    copy[tr][tc] = piece;


    /* CASTLING */

    if (
        (piece === "♔" || piece === "♚") &&
        Math.abs(tc-fc) === 2
    ) {

        if (tc === 6) {

            copy[tr][5] = copy[tr][7];
            copy[tr][7] = "";

        }
        else {

            copy[tr][3] = copy[tr][0];
            copy[tr][0] = "";

        }

    }


    return copy;

}


/* =========================================================
   LEGAL MOVES
   ========================================================= */

function getLegalMoves(row,col) {

    const piece = board[row][col];

    if (piece === "") {
        return [];
    }


    const white = isWhite(piece);


    let moves =
        getPseudoMoves(
            board,
            row,
            col,
            enPassantRow,
            enPassantCol
        );


    const castle =
        getCastlingMoves(
            board,
            row,
            col
        );


    moves =
        moves.concat(castle);


    const legal = [];


    for (let move of moves) {

        const testBoard =
            simulateMove(
                board,
                row,
                col,
                move.row,
                move.col,
                enPassantRow,
                enPassantCol,
                enPassantPawnRow,
                enPassantPawnCol
            );


        if (
            !isInCheck(
                testBoard,
                white
            )
        ) {

            legal.push(move);

        }

    }


    return legal;

}


/* =========================================================
   MAKE MOVE
   ========================================================= */

function makeMove(
    fr,
    fc,
    tr,
    tc,
    move
) {

    const piece = board[fr][fc];


    /* EN PASSANT */

    if (move.enPassant) {

        if (enPassantPawnRow !== null) {

            board[
                enPassantPawnRow
            ][
                enPassantPawnCol
            ] = "";

        }

    }


    board[fr][fc] = "";

    board[tr][tc] = piece;


    /* CASTLING */

    if (
        (piece === "♔" || piece === "♚") &&
        Math.abs(tc-fc) === 2
    ) {

        if (tc === 6) {

            board[tr][5] = board[tr][7];
            board[tr][7] = "";

        }
        else {

            board[tr][3] = board[tr][0];
            board[tr][0] = "";

        }

    }


    /* KING RIGHTS */

    if (piece === "♔") {
        whiteKingMoved = true;
    }

    if (piece === "♚") {
        blackKingMoved = true;
    }


    /* ROOK RIGHTS */

    if (piece === "♖") {

        if (fr === 7 && fc === 0) {
            whiteRookAMoved = true;
        }

        if (fr === 7 && fc === 7) {
            whiteRookHMoved = true;
        }

    }


    if (piece === "♜") {

        if (fr === 0 && fc === 0) {
            blackRookAMoved = true;
        }

        if (fr === 0 && fc === 7) {
            blackRookHMoved = true;
        }

    }


    /* ROOK CAPTURE */

    if (tr === 7 && tc === 0) {
        whiteRookAMoved = true;
    }

    if (tr === 7 && tc === 7) {
        whiteRookHMoved = true;
    }

    if (tr === 0 && tc === 0) {
        blackRookAMoved = true;
    }

    if (tr === 0 && tc === 7) {
        blackRookHMoved = true;
    }


    /* RESET EP */

    enPassantRow = null;
    enPassantCol = null;

    enPassantPawnRow = null;
    enPassantPawnCol = null;


    /* CREATE EP */

    if (
        piece === "♙" &&
        fr === 6 &&
        tr === 4
    ) {

        enPassantRow = 5;
        enPassantCol = fc;

        enPassantPawnRow = tr;
        enPassantPawnCol = tc;

    }


    if (
        piece === "♟" &&
        fr === 1 &&
        tr === 3
    ) {

        enPassantRow = 2;
        enPassantCol = fc;

        enPassantPawnRow = tr;
        enPassantPawnCol = tc;

    }


    /* PROMOTION */

    if (
        piece === "♙" &&
        tr === 0
    ) {

        promotionRow = tr;
        promotionCol = tc;
        promotionWhite = true;

        document.getElementById(
            "promotionMenu"
        ).style.display = "block";

        document.getElementById(
            "promotionPlayer"
        ).textContent =
            "Choose White Promotion";

        return true;

    }


    if (
        piece === "♟" &&
        tr === 7
    ) {

        promotionRow = tr;
        promotionCol = tc;
        promotionWhite = false;

        document.getElementById(
            "promotionMenu"
        ).style.display = "block";

        document.getElementById(
            "promotionPlayer"
        ).textContent =
            "Choose Black Promotion";

        return true;

    }


    return false;

}


/* =========================================================
   PROMOTION
   ========================================================= */

function finishPromotion(type) {

    const whitePieces = {

        queen:"♕",
        rook:"♖",
        bishop:"♗",
        knight:"♘"

    };


    const blackPieces = {

        queen:"♛",
        rook:"♜",
        bishop:"♝",
        knight:"♞"

    };


    if (promotionWhite) {

        board[promotionRow][promotionCol] =
            whitePieces[type];

    }
    else {

        board[promotionRow][promotionCol] =
            blackPieces[type];

    }


    document.getElementById(
        "promotionMenu"
    ).style.display = "none";


    promotionRow = null;
    promotionCol = null;


    whiteTurn = !whiteTurn;

    selectedRow = null;
    selectedCol = null;


    drawBoard();

    updateTurnText();

    checkGameStatus();


    if (!gameOver && !whiteTurn) {

        computerMove();

    }

}


/* =========================================================
   ANY LEGAL MOVE
   ========================================================= */

function hasAnyLegalMove(white) {

    for (let r=0; r<8; r++) {

        for (let c=0; c<8; c++) {

            const piece = board[r][c];

            if (
                piece !== "" &&
                isWhite(piece) === white
            ) {

                if (
                    getLegalMoves(r,c).length > 0
                ) {

                    return true;

                }

            }

        }

    }


    return false;

}


/* =========================================================
   GAME STATUS
   ========================================================= */

function checkGameStatus() {

    const currentWhite = whiteTurn;


    const check =
        isInCheck(
            board,
            currentWhite
        );


    const hasMove =
        hasAnyLegalMove(
            currentWhite
        );


    const status =
        document.getElementById(
            "statusText"
        );


    if (!hasMove && check) {

        gameOver = true;

        stopClock();


        if (currentWhite) {

            status.textContent =
                "☠️ Checkmate! Computer wins!";

        }
        else {

            status.textContent =
                "☠️ Checkmate! You win!";

        }

        return;

    }


    if (!hasMove && !check) {

        gameOver = true;

        stopClock();

        status.textContent =
            "🤝 Stalemate! Draw!";

        return;

    }


    if (check) {

        if (currentWhite) {

            status.textContent =
                "⚠️ You are in CHECK!";

        }
        else {

            status.textContent =
                "⚠️ Computer is in CHECK!";

        }

    }
    else {

        status.textContent = "";

    }

}


/* =========================================================
   PLAYER CLICK
   ========================================================= */

function squareClicked(row,col) {

    if (gameOver) {
        return;
    }


    if (computerThinking) {
        return;
    }


    if (!whiteTurn) {
        return;
    }


    if (promotionRow !== null) {
        return;
    }


    const piece = board[row][col];


    /* SELECT */

    if (selectedRow === null) {

        if (
            piece !== "" &&
            isWhite(piece)
        ) {

            selectedRow = row;
            selectedCol = col;

            highlightSelected();

        }

        return;

    }


    /* DESELECT */

    if (
        selectedRow === row &&
        selectedCol === col
    ) {

        selectedRow = null;
        selectedCol = null;

        highlightSelected();

        return;

    }


    /* RESELECT */

    if (
        piece !== "" &&
        isWhite(piece)
    ) {

        selectedRow = row;
        selectedCol = col;

        highlightSelected();

        return;

    }


    const moves =
        getLegalMoves(
            selectedRow,
            selectedCol
        );


    const move =
        moves.find(
            m =>
                m.row === row &&
                m.col === col
        );


    if (!move) {

        document.getElementById(
            "messageText"
        ).textContent =
            "❌ Illegal move";

        return;

    }


    document.getElementById(
        "messageText"
    ).textContent = "";


    const wasPromotion =
        makeMove(
            selectedRow,
            selectedCol,
            row,
            col,
            move
        );


    selectedRow = null;
    selectedCol = null;


    if (wasPromotion) {

        drawBoard();

        return;

    }


    whiteTurn = false;


    drawBoard();

    updateTurnText();

    checkGameStatus();


    if (!gameOver) {

        computerMove();

    }

}


/* =========================================================
   COMPUTER AI
   ========================================================= */

function computerMove() {

    if (gameOver || whiteTurn) {
        return;
    }


    computerThinking = true;


    document.getElementById(
        "messageText"
    ).textContent =
        "🤖 Computer is thinking...";


    setTimeout(function() {

        if (gameOver) {

            computerThinking = false;
            return;

        }


        const moves = [];


        /* FIND ALL BLACK LEGAL MOVES */

        for (let r=0; r<8; r++) {

            for (let c=0; c<8; c++) {

                const piece = board[r][c];

                if (
                    piece !== "" &&
                    !isWhite(piece)
                ) {

                    const legal =
                        getLegalMoves(r,c);


                    legal.forEach(move => {

                        moves.push({

                            fromRow:r,
                            fromCol:c,
                            toRow:move.row,
                            toCol:move.col,
                            move:move

                        });

                    });

                }

            }

        }


        if (moves.length === 0) {

            computerThinking = false;

            checkGameStatus();

            return;

        }


        /* SCORE MOVES */

        const scoredMoves =
            moves.map(item => {

                let score = Math.random() * 5;


                const target =
                    board[
                        item.toRow
                    ][
                        item.toCol
                    ];


                /* CAPTURE BONUS */

                if (target !== "") {

                    score +=
                        pieceValue(target) * 10;

                }


                /* CHECK BONUS */

                const testBoard =
                    simulateMove(
                        board,
                        item.fromRow,
                        item.fromCol,
                        item.toRow,
                        item.toCol,
                        enPassantRow,
                        enPassantCol,
                        enPassantPawnRow,
                        enPassantPawnCol
                    );


                if (
                    isInCheck(
                        testBoard,
                        true
                    )
                ) {

                    score += 50;

                }


                /* CASTLING BONUS */

                if (item.move.castle) {

                    score += 20;

                }


                return {
                    item:item,
                    score:score
                };

            });


        scoredMoves.sort(
            (a,b) => b.score - a.score
        );


        const best =
            scoredMoves[0].item;


        const wasPromotion =
            makeMove(
                best.fromRow,
                best.fromCol,
                best.toRow,
                best.toCol,
                best.move
            );


        computerThinking = false;


        if (wasPromotion) {

            /* COMPUTER ALWAYS CHOOSES QUEEN */

            board[
                promotionRow
            ][
                promotionCol
            ] =
                promotionWhite
                ? "♕"
                : "♛";


            document.getElementById(
                "promotionMenu"
            ).style.display = "none";


            promotionRow = null;
            promotionCol = null;


            whiteTurn = true;

        }
        else {

            whiteTurn = true;

        }


        document.getElementById(
            "messageText"
        ).textContent = "";


        drawBoard();

        updateTurnText();

        checkGameStatus();


    }, 700);

}


/* =========================================================
   PIECE VALUES
   ========================================================= */

function pieceValue(piece) {

    switch(piece) {

        case "♙":
        case "♟":
            return 1;

        case "♘":
        case "♞":
            return 3;

        case "♗":
        case "♝":
            return 3;

        case "♖":
        case "♜":
            return 5;

        case "♕":
        case "♛":
            return 9;

        case "♔":
        case "♚":
            return 100;

        default:
            return 0;

    }

}


/* =========================================================
   TURN TEXT
   ========================================================= */

function updateTurnText() {

    const turn =
        document.getElementById(
            "turnText"
        );


    if (whiteTurn) {

        turn.textContent =
            "Your Turn ♔";

    }
    else {

        turn.textContent =
            "Computer's Turn ♚";

    }

}


/* =========================================================
   CLOCK
   ========================================================= */

function formatTime(seconds) {

    const minutes =
        Math.floor(seconds / 60);

    const secs =
        seconds % 60;


    return (
        String(minutes).padStart(2,"0")
        +
        ":"
        +
        String(secs).padStart(2,"0")
    );

}


function updateClocks() {

    document.getElementById(
        "whiteClock"
    ).textContent =
        formatTime(whiteTime);


    document.getElementById(
        "blackClock"
    ).textContent =
        formatTime(blackTime);

}


function stopClock() {

    if (clockInterval !== null) {

        clearInterval(clockInterval);

        clockInterval = null;

    }

}


function startClock() {

    stopClock();


    clockInterval =
        setInterval(function() {

            if (gameOver) {

                stopClock();

                return;

            }


            if (whiteTurn) {

                whiteTime--;

            }
            else {

                blackTime--;

            }


            updateClocks();


            if (whiteTime <= 0) {

                whiteTime = 0;

                gameOver = true;

                stopClock();


                document.getElementById(
                    "statusText"
                ).textContent =
                    "⏱️ Time's up! Computer wins!";

                return;

            }


            if (blackTime <= 0) {

                blackTime = 0;

                gameOver = true;

                stopClock();


                document.getElementById(
                    "statusText"
                ).textContent =
                    "⏱️ Time's up! You win!";

                return;

            }

        },1000);

}


/* =========================================================
   RESTART
   ========================================================= */

function restartGame() {

    board =
        copyBoard(startingBoard);


    whiteTurn = true;

    selectedRow = null;
    selectedCol = null;

    gameOver = false;

    computerThinking = false;


    whiteKingMoved = false;
    blackKingMoved = false;

    whiteRookAMoved = false;
    whiteRookHMoved = false;

    blackRookAMoved = false;
    blackRookHMoved = false;


    enPassantRow = null;
    enPassantCol = null;

    enPassantPawnRow = null;
    enPassantPawnCol = null;


    promotionRow = null;
    promotionCol = null;


    whiteTime = 10 * 60;
    blackTime = 10 * 60;


    document.getElementById(
        "promotionMenu"
    ).style.display = "none";


    document.getElementById(
        "messageText"
    ).textContent = "";


    document.getElementById(
        "statusText"
    ).textContent = "";


    updateTurnText();

    updateClocks();

    drawBoard();

    startClock();

}


/* =========================================================
   START
   ========================================================= */

updateTurnText();

updateClocks();

drawBoard();

startClock();

</script>

</body>
</html>
"""

display(HTML(html))

SyntaxError: f-string: expecting '!', or ':', or '}' (2223442318.py, line 985)